# Suite PHY — The physical layout

`Physical` defines the coordinates physical layout gives the logical layout's points, the container layout's frames and a diagram's viewport, each an annotation owned by what it annotates. mbse-basis [CAP-241: physical layout schemas](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-241); its components, which place them, are TypeScript's only.

In [ ]:
from support import Bench, text
from mbse.Schemas.Framework import JSON, Paths, Validators
from mbse.Diagrams.Framework import Container, Logical, Physical

b = Bench()
b.store.transaction(Physical.install)
L, P = b.store.Logical, b.store.Physical


def at(x, y):
    return lambda v: v.x(x).y(y)


def size(width, height):
    return lambda v: v.width(width).height(height)

<a id="PHY-01"></a>
## PHY-01 · The install, after Logical's and Container's, defines the physical layout's schemas, each valid, and run again writes nothing

In [ ]:
NAMES = ['Physical.OfPoint2', 'Physical.OfSize', 'Physical.Placement', 'Physical.OfPlacement', 'Physical.GridFrame', 'Physical.CellFrame', 'Physical.OfFrame', 'Physical.Viewport', 'Physical.OfViewport']
ANNOTATED = {"Logical.OfPoint": "placement", "Container.OfGrid": "frame", "Container.OfCell": "frame",
             "Logical.OfDiagram": "viewport"}
schemas = [b.store.registered(name) for name in NAMES]
assert [schema.validate() for schema in schemas] == [[]] * len(NAMES)
b.store.transaction(Physical.install)
assert [b.store.registered(name) for name in NAMES] == schemas
assert all(member in b.store.registered(name).adjacencies for name, member in ANNOTATED.items())

<a id="PHY-02"></a>
## PHY-02 · A point's placement is owned by the point: its absolute and relative coordinates and its direction, and its path the point's

In [ ]:
assert b.grid('cpu', 2) == '[{"kind":"grid","shape":"cpu","span":2,"grows":"rows","cell_rotation":0.0}]'
assert b.child('cpu', 'a') == '[{"kind":"place","shape":"a","order":0,"to":[0,0]},{"kind":"extents","rows":1,"columns":1}]'
a = b.shape("cpu.a")
with b.store.transaction():
    L.OfShape(a, lambda s: s.origin(lambda x: x.insert(lambda e: e.point(lambda p: p))))
(o,) = [e.point for e in a.origin]
with b.store.transaction():
    L.OfPoint(o, lambda p: p.placement(lambda x: x.insert(lambda e: e.placement(
        lambda q: q.absolute(at(40.0, 30.0)).relative(at(40.0, 30.0)).direction(0.0)))))
(placement,) = [e.placement for e in o.placement]
path = lambda x: JSON.ToJSON(b.store)(b.store.Paths.OfRoot.as_schema(), Paths.of(x))
assert text([placement.absolute.x.get(), placement.absolute.y.get(), placement.relative.x.get(), placement.direction.get()]) == '[40.0,30.0,40.0,0.0]'
assert text([e.point is o for e in placement.point] == [True]) == 'true'
assert text(path(placement)[len(path(o)) - 13:]) == '"{\\"i\\": \\"placement\\", \\"o\\": \\"point\\", \\"properties\\": {}, \\"next\\": {\\"target\\": {}}}}}}}}}}}}}"'
assert text(JSON.ToJSON(b.store)(P.OfPlacement.as_schema(), placement)) == '"{\\"mutations\\": [{\\"upsert\\": {\\"exists\\": [{\\"symbols\\": {\\"0\\": {\\"schema\\": [{\\"$ref\\": {\\"singleton\\": \\".\\", \\"next\\": {\\"entry\\": {\\"i\\": \\"schemas\\", \\"o\\": \\"\\", \\"properties\\": {\\"name\\": \\"Physical\\"}, \\"next\\": {\\"entry\\": {\\"i\\": \\"schemas\\", \\"o\\": \\"\\", \\"properties\\": {\\"name\\": \\"OfPlacement\\"}, \\"next\\": {\\"target\\": {}}}}}}}}]}}, \\"spec\\": [{\\"0\\": [{\\"as_property\\": {\\"name\\": \\"absolute\\", \\"mutations\\": [{\\"set\\": {\\"value\\": {\\"x\\": 40.0, \\"y\\": 30.0}}}]}}, {\\"as_property\\": {\\"name\\": \\"relative\\", \\"mutations\\": [{\\"set\\": {\\"value\\": {\\"x\\": 40.0, \\"y\\": 30.0}}}]}}, {\\"as_property\\": {\\"name\\": \\"direction\\", \\"mutations\\": [{\\"set\\": {\\"value\\": 0.0}}]}}, {\\"as_adjacency\\": {\\"name\\": \\"point\\", \\"mutations\\": [{\\"insert\\": {\\"entries\\": [{\\"mutations\\": [{\\"as_link\\": {\\"name\\": \\"point\\", \\"mutations\\": [{\\"set\\": {\\"ref\\": {\\"path\\": {\\"singleton\\": \\"Logical.Diagrams\\", \\"next\\": {\\"entry\\": {\\"i\\": \\"diagrams\\", \\"o\\": \\"directory\\", \\"properties\\": {\\"name\\": \\"overview\\"}, \\"next\\": {\\"entry\\": {\\"i\\": \\"shapes\\", \\"o\\": \\"diagram\\", \\"properties\\": {\\"name\\": \\"cpu\\"}, \\"next\\": {\\"entry\\": {\\"i\\": \\"children\\", \\"o\\": \\"container\\", \\"properties\\": {\\"name\\": \\"a\\"}, \\"next\\": {\\"entry\\": {\\"i\\": \\"origin\\", \\"o\\": \\"origin_of\\", \\"properties\\": {}, \\"next\\": {\\"target\\": {}}}}}}}}}}}}}}]}}]}]}}]}}]}]}]}}]}"'

<a id="PHY-03"></a>
## PHY-03 · A grid's and a cell's frame, and a diagram's viewport, each owned by what it annotates

In [ ]:
((entry,),) = [list(b.shape("cpu").grid)]
grid = entry.grid
(cell,) = [e.cell for e in grid.cells]
with b.store.transaction():
    b.store.Container.OfGrid(grid, lambda g: g.frame(lambda x: x.insert(lambda e: e.frame(
        lambda f: f.absolute(at(0.0, 0.0)).relative(at(0.0, 0.0)).size(size(200.0, 80.0))))))
    b.store.Container.OfCell(cell, lambda c: c.frame(lambda x: x.insert(lambda e: e.frame(
        lambda f: f.absolute(at(0.0, 0.0)).relative(at(0.0, 0.0)).size(size(100.0, 80.0))))))
    L.OfDiagram(b.overview, lambda d: d.viewport(lambda x: x.insert(lambda e: e.viewport(
        lambda v: v.size(size(800.0, 600.0))))))
(frame,) = [e.frame for e in cell.frame]
assert text([[e.frame.size.width.get() for e in grid.frame], frame.size.width.get(), [e.viewport.size.height.get() for e in b.overview.viewport]]) == '[[200.0],100.0,[600.0]]'
assert text(Validators.Validate(b.store).Reachable(L.OfDiagram, b.overview)) == '[]'

<a id="PHY-04"></a>
## PHY-04 · One placement per point, as the validator reports

In [ ]:
with b.store.transaction():
    L.OfPoint(o, lambda p: p.placement(lambda x: x.insert(lambda e: e.placement(lambda q: q.direction(90.0)))))
assert text(Validators.Validate(b.store).Reachable(L.OfDiagram, b.overview)) == '["unique(placement) violated: entries agreeing on [\'point\'] differ on [\'placement\']"]'

<a id="PHY-05"></a>
## PHY-05 · An annotation goes with what it annotates: a child removed takes its points' placements, and its cell's frame

In [ ]:
assert b.unchild('cpu', 'a') == '[{"kind":"leave","shape":"a","from":[0,0]},{"kind":"extents","rows":0,"columns":0}]'
assert text([b.store.holds(placement), b.store.holds(frame), b.store.holds(grid)]) == '[false,false,false]'